In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.model_selection import train_test_split

In [35]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("flight_delay_base")

2026/08/13 15:28:41 INFO mlflow.tracking.fluent: Experiment with name 'flight_delay_base' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1786649321783, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786649321783, lifecycle_stage='active', name='flight_delay_base', tags={}, trace_location=None, workspace='default'>

In [2]:
%pwd

'/Users/nicholasstanfield/Desktop/flight-delay/notebooks'

In [3]:
df = pd.read_csv("../data/processed/flight_data_2025.csv")
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,AIRLINE
0,1,1,1,3,AUS,ORD,545,830,0.0,165.0,977.0,American Airlines Inc.
1,1,1,7,2,PBI,DFW,620,842,0.0,202.0,1102.0,American Airlines Inc.
2,1,1,26,7,LAS,DEN,920,1219,0.0,119.0,628.0,United Air Lines Inc.
3,1,1,17,5,ICT,ATL,1750,2105,0.0,135.0,782.0,Delta Air Lines Inc.
4,1,1,14,2,CHS,EWR,1930,2129,0.0,119.0,628.0,United Air Lines Inc.


In [8]:
df["ARR_DEL15"] = df["ARR_DEL15"].astype(int)

In [9]:
X = df.drop("ARR_DEL15",axis=1)
y = df["ARR_DEL15"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [14]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((288000, 11), (72000, 11), (288000,), (72000,))

In [20]:
categorical_columns = X_train.select_dtypes(include=["object"]).columns.to_list()

numeric_columns = X_train.select_dtypes(include=["int","float"]).columns.to_list()

In [24]:
X["ORIGIN"].nunique(),X["DEST"].nunique(),X["AIRLINE"].nunique()

(351, 350, 14)

In [29]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


cat_pipeline = Pipeline([("one_hot_encoder", OneHotEncoder(handle_unknown='ignore'))])
num_pipeline = Pipeline([("scale", StandardScaler())])


preprocessing = ColumnTransformer([
    ("num", num_pipeline, numeric_columns),
    ("cat", cat_pipeline, categorical_columns),
])

In [33]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

(ARR_DEL15
 0    0.778365
 1    0.221635
 Name: proportion, dtype: float64,
 ARR_DEL15
 0    0.774958
 1    0.225042
 Name: proportion, dtype: float64)

In [37]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

dummy_classifier = make_pipeline(
    preprocessing,
    DummyClassifier(strategy="prior")
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

In [38]:
import mlflow.sklearn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="dummy_classifier"):

    cv_results = cross_validate(
        dummy_classifier,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_balanced_accuracy_mean": cv_results["test_balanced_accuracy"].mean(),
        "cv_precision_mean": cv_results["test_precision"].mean(),
        "cv_recall_mean": cv_results["test_recall"].mean(),
        "cv_f1_mean": cv_results["test_f1"].mean(),
    }

    mlflow.log_metrics(metrics)

metrics

🏃 View run dummy_classifier at: http://localhost:5001/#/experiments/1/runs/7e3a8eb6310344b3865927fe06ccb0a6
🧪 View experiment at: http://localhost:5001/#/experiments/1


/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

{'cv_accuracy_mean': np.float64(0.7783645833333332),
 'cv_balanced_accuracy_mean': np.float64(0.5),
 'cv_precision_mean': np.float64(0.0),
 'cv_recall_mean': np.float64(0.0),
 'cv_f1_mean': np.float64(0.0)}

In [39]:
from sklearn.linear_model import LogisticRegression

logistic_regression = make_pipeline(
    preprocessing,
    LogisticRegression(
        solver="saga",
        max_iter=1000,
        random_state=42
    )
)

with mlflow.start_run(run_name="logistic_regression"):

    cv_results = cross_validate(
        logistic_regression,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_balanced_accuracy_mean": cv_results["test_balanced_accuracy"].mean(),
        "cv_precision_mean": cv_results["test_precision"].mean(),
        "cv_recall_mean": cv_results["test_recall"].mean(),
        "cv_f1_mean": cv_results["test_f1"].mean(),

        "cv_accuracy_std": cv_results["test_accuracy"].std(),
        "cv_balanced_accuracy_std": cv_results["test_balanced_accuracy"].std(),
        "cv_precision_std": cv_results["test_precision"].std(),
        "cv_recall_std": cv_results["test_recall"].std(),
        "cv_f1_std": cv_results["test_f1"].std(),
    }

    mlflow.log_metrics(metrics)

metrics

/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nicholasstanfield/Desktop/flight-delay/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

🏃 View run logistic_regression at: http://localhost:5001/#/experiments/1/runs/0efef8d48ae845afb61da8887ed673e1
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.7779583333333333),
 'cv_balanced_accuracy_mean': np.float64(0.502450751331057),
 'cv_precision_mean': np.float64(0.4463576875379619),
 'cv_recall_mean': np.float64(0.007582516225167843),
 'cv_f1_mean': np.float64(0.014911094499857691),
 'cv_accuracy_std': np.float64(0.00019131912936047588),
 'cv_balanced_accuracy_std': np.float64(0.0002716850097362152),
 'cv_precision_std': np.float64(0.024619089835912787),
 'cv_recall_std': np.float64(0.0005036724710059672),
 'cv_f1_std': np.float64(0.0009834430993230548)}

## TODO

* Use MLFlow for each experiment
* Split data into X, y
* Split the data into train-test (use CV so no need for a validation set)
* One hot encode all categorical columns
* Explore some feature engineering such as route create, speed column (distance over time)
* Try the following models:
    * Logistic Regression
    * SVM
    * Decision Trees
    * Random Forest
    * Gradient Boosting
    * XGBoost (and similiar, e.g. LightGBM, CatBoost)
    * Neural Networks (through sklearn)
* Export final model into a model folder (DO NOT DO THIS IN THIS NOTEBOOK TAKE THE BEST MODEL AND WRITE THE CODE IN A model_pipeline.py in src
    * Not just the pipeline but the preprocessing as well so the prediction can be easily fed in